# Memory in AI Systems

## Chapter 34: Wiring Memory into the Tutor Loop

Read this chapter: [https://books.vizuara.ai/read/memory-workshop/wiring-memory-into-the-tutor](https://books.vizuara.ai/read/memory-workshop/wiring-memory-into-the-tutor)

---

**Generated file.** Extracted automatically from the book's own text. Do not hand-edit; edit the book instead.

Code cells appear in the order the blocks appear in the chapter, and may assume names defined in earlier cells or earlier chapters. Nothing here has been executed.

In [ ]:
# --- block 1 ---
from openai import OpenAI

client = OpenAI()

SYSTEM = "You are a patient AI tutor. Explain simply and check for understanding."

def tutor_reply(user_msg):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_msg},
    ]
    resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return resp.choices[0].message.content

while True:
    user_msg = input("you › ")
    print("tutor ›", tutor_reply(user_msg))

In [ ]:
# --- block 2 ---
from mem0 import Memory

memory = Memory()          # local vector store + LLM extractor, out of the box

def recall(user_msg, user_id):
    hits = memory.search(query=user_msg, user_id=user_id, limit=5)
    return [m["memory"] for m in hits["results"]]

In [ ]:
# --- block 3 ---
def build_messages(user_msg, memories):
    memory_block = "\n".join(f"- {m}" for m in memories)
    system = SYSTEM
    if memory_block:
        system += f"\n\nWhat you remember about this learner:\n{memory_block}"
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]

In [ ]:
# --- block 4 ---
def tutor_turn(user_msg, user_id):
    memories = recall(user_msg, user_id)                 # 1. READ
    messages = build_messages(user_msg, memories)        # 2. AUGMENT
    reply = client.chat.completions.create(              # 3. GENERATE
        model="gpt-4o-mini", messages=messages
    ).choices[0].message.content
    memory.add(                                          # 4. WRITE
        [{"role": "user", "content": user_msg},
         {"role": "assistant", "content": reply}],
        user_id=user_id,
    )
    return reply

In [ ]:
# --- block 5 ---
from openai import OpenAI
from mem0 import Memory

client = OpenAI()
memory = Memory()
SYSTEM = "You are a patient AI tutor. Explain simply and check for understanding."

def tutor_turn(user_msg, user_id):
    hits = memory.search(query=user_msg, user_id=user_id, limit=5)   # READ
    mems = [m["memory"] for m in hits["results"]]
    system = SYSTEM + ("\n\nWhat you remember about this learner:\n" +
                       "\n".join(f"- {m}" for m in mems) if mems else "")
    reply = client.chat.completions.create(                          # AUGMENT + GENERATE
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user_msg}],
    ).choices[0].message.content
    memory.add([{"role": "user", "content": user_msg},               # WRITE
                {"role": "assistant", "content": reply}], user_id=user_id)
    return reply

while True:
    print("tutor ›", tutor_turn(input("you › "), user_id="aditya"))